# 01. 인프라 배포 및 AI Search 파이프라인 설정

이 노트북에서는 Bicep으로 전체 인프라를 배포하고, Azure AI Search 파이프라인(인덱서 + 스킬셋)을 설정합니다.

## 지원 리전
| 리전 | Bicep 경로 | 특이사항 |
|------|-----------|----------|
| **Sweden Central** | `infra/sweden/` | DI 동일 리전, 단일 RG |
| **Korea Central** | `infra/korea/` | DI East US 2 Cross-Region PE, 2개 RG |

## 배포 리소스
| 리소스 | SKU | 용도 |
|--------|-----|------|
| Virtual Network (10.0.0.0/16) | - | Private Network 격리 |
| Storage Account | Standard LRS | 크롤링 데이터 저장 (Private) |
| Azure AI Services | S0 | GPT-5.4, text-embedding-3-large |
| Document Intelligence | S0 | PDF 레이아웃 분석 |
| Azure AI Search | **Standard (S1)** | 벡터 + 시맨틱 하이브리드 검색 |
| Function App (EP1) | Elastic Premium | Python 크롤러 (VNet Integration) |
| Logic App | Consumption | 크롤 스케줄러 (매일 21:00 UTC) |
| Private Endpoints × 4+ | - | 모든 서비스 프라이빗 접근 |
| Shared Private Links × 3 | - | AI Search 아웃바운드 |

## 노트북 흐름
```
1. 리전 선택 + 사전 요구사항 확인
2. Bicep 배포 (선택 리전, Private Network)
3. Shared Private Link 배포 + 승인
4. Function App 코드 배포 (crawl-function/)
5. AI Search 파이프라인 설정 (Index + Skillset + Indexer)
6. .env 파일 생성
7. 배포 검증
```

## 1. 리전 선택 + 사전 요구사항 확인

아래 셀에서 배포할 리전을 선택합니다. `DEPLOY_REGION`을 `"sweden"` 또는 `"korea"`로 설정하세요.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  배포 리전 선택 — "sweden" 또는 "korea"       ║
# ╚══════════════════════════════════════════════════╝
DEPLOY_REGION = "korea"   # ← 변경: "korea"

assert DEPLOY_REGION in ("sweden", "korea"), f"Invalid region: {DEPLOY_REGION}"
print(f"✅ 배포 리전: {DEPLOY_REGION}")

In [ ]:
# Azure CLI 로그인 확인
!az account show --output table

In [ ]:
import subprocess
import json
from getpass import getpass

# 구독 ID
result = subprocess.run(
    ["az", "account", "show", "--query", "id", "--output", "tsv"],
    capture_output=True, text=True
)
SUBSCRIPTION_ID = result.stdout.strip()

# ── 리전별 설정 ──
# ※ 양쪽 Bicep 모두 GPT 파라미터명: gptDeploymentName / gptModelVersion
REGION_CONFIG = {
    "sweden": {
        "location": "swedencentral",
        "rg_name": "rg-rag-indexing-lab-swc",
        "deployment_name": "rag-indexing-lab-swc",
        "bicep_path": "../infra/sweden/main.bicep",
        "spl_bicep_path": "../infra/sweden/modules/ai-search-spl.bicep",
        "has_di_rg": False,
    },
    "korea": {
        "location": "koreacentral",
        "rg_name": "rg-rag-indexing-lab-krc",
        "deployment_name": "rag-indexing-lab-krc",
        "bicep_path": "../infra/korea/main.bicep",
        "spl_bicep_path": "../infra/korea/modules/ai-search-spl.bicep",
        "has_di_rg": True,
        "di_rg_name": "rg-rag-indexing-lab-eus2",
    },
}

cfg = REGION_CONFIG[DEPLOY_REGION]
RG_NAME = cfg["rg_name"]
LOCATION = cfg["location"]
DEPLOYMENT_NAME = cfg["deployment_name"]
BICEP_PATH = cfg["bicep_path"]

# JumpVM 설정
JUMPVM_ADMIN_PASSWORD = getpass("JumpVM admin password: ")
JUMPVM_ENTRA_USER_OBJECT_IDS = "['4cb458b5-ef0c-403d-8162-c39855018986']"

print(f"Region:      {DEPLOY_REGION} ({LOCATION})")
print(f"RG:          {RG_NAME}")
print(f"Bicep:       {BICEP_PATH}")
print(f"Deployment:  {DEPLOYMENT_NAME}")
print(f"Subscription: {SUBSCRIPTION_ID}")

In [ ]:
# 필수 도구 확인
!az bicep version
!func --version  # Azure Functions Core Tools

## 2. Bicep 배포

Private Network(VNet + Private Endpoints) 전체 인프라를 한 번에 배포합니다.  
약 10~15분 소요됩니다.

- **Sweden Central**: 단일 RG, DI 동일 리전
- **Korea Central**: 메인 RG(Korea Central) + DI RG(East US 2), Cross-Region PE

In [ ]:
# 리전별 Bicep 배포 (~10분 소요)
# ※ 양쪽 Bicep 모두 동일 파라미터명: gptDeploymentName / gptModelVersion
deploy_cmd = [
    "az", "deployment", "sub", "create",
    "--location", LOCATION,
    "--template-file", BICEP_PATH,
    "--name", DEPLOYMENT_NAME,
    "--parameters", f"resourceGroupName={RG_NAME}",
    "--parameters", f"location={LOCATION}",
    "--parameters", "embeddingDeploymentName=text-embedding-3-large",
    "--parameters", "gptDeploymentName=gpt-5.4",
    "--parameters", "gptModelVersion=2026-03-05",
    "--parameters", "searchSku=standard",
    "--parameters", "blobContainerName=raw-documents",
    "--parameters", "jumpvmAdminUsername=azureadmin",
    "--parameters", f"jumpvmAdminPassword={JUMPVM_ADMIN_PASSWORD}",
    "--parameters", f"jumpvmEntraUserObjectIds={JUMPVM_ENTRA_USER_OBJECT_IDS}",
    "--output", "table",
]

print(f"배포 시작: {DEPLOY_REGION} ({LOCATION})")
print(f"Bicep: {BICEP_PATH}")
result = subprocess.run(deploy_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
# 배포 출력값 파싱
result = subprocess.run(
    ["az", "deployment", "sub", "show",
     "--name", DEPLOYMENT_NAME,
     "--query", "properties.outputs",
     "--output", "json"],
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)

print(f"=== 배포된 리소스 ({DEPLOY_REGION}) ===")
for key, val in outputs.items():
    print(f"  {key}: {val['value']}")

# GPT deployment name (양쪽 Bicep 동일: gptDeploymentName)
GPT_DEPLOY_NAME = outputs.get('gptDeploymentName', {}).get('value', 'gpt-5.4')

## 3. Shared Private Link 배포 + 승인

### 3a. SPL 배포
AI Search가 아웃바운드로 Storage, AI Services, Document Intelligence에 접근하기 위한  
Shared Private Link를 배포합니다. **초회 1회만 실행**합니다.

> SPL이 Approved 상태가 되면 재배포 시 ARM이 "conflicting update" 오류를 발생시키므로 메인 Bicep과 분리

In [ ]:
SEARCH_NAME = outputs['aiSearchName']['value']
STORAGE_NAME = outputs['storageAccountName']['value']
AI_SERVICES_NAME = outputs['openaiAccountName']['value']

# suffix 추출 (storageAccountName = 'stragi<suffix>')
_suffix = STORAGE_NAME.replace('stragi', '')
DI_NAME = f"di-ragi-{_suffix}"

# DI RG — Sweden: 메인 RG, Korea: 별도 RG (East US 2)
DI_RG = cfg.get("di_rg_name", RG_NAME) if cfg["has_di_rg"] else RG_NAME

# 각 리소스 ID 조회
def get_resource_id(cmd_args):
    r = subprocess.run(cmd_args, capture_output=True, text=True)
    return r.stdout.strip()

STORAGE_RESOURCE_ID = get_resource_id([
    "az", "storage", "account", "show",
    "--name", STORAGE_NAME, "--resource-group", RG_NAME,
    "--query", "id", "--output", "tsv"
])
AI_SERVICES_RESOURCE_ID = get_resource_id([
    "az", "cognitiveservices", "account", "show",
    "--name", AI_SERVICES_NAME, "--resource-group", RG_NAME,
    "--query", "id", "--output", "tsv"
])
DI_RESOURCE_ID = get_resource_id([
    "az", "cognitiveservices", "account", "show",
    "--name", DI_NAME, "--resource-group", DI_RG,
    "--query", "id", "--output", "tsv"
])

# SPL Bicep 배포 (초회 1회만)
spl_bicep = cfg["spl_bicep_path"]
print(f"SPL 배포: {spl_bicep}")
print(f"  → spl-blob:        {STORAGE_NAME}")
print(f"  → spl-aiservices:  {AI_SERVICES_NAME}")
print(f"  → spl-docintel:    {DI_NAME} (RG: {DI_RG})")

spl_result = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", RG_NAME,
    "--template-file", spl_bicep,
    "--parameters", f"searchServiceName={SEARCH_NAME}",
    "--parameters", f"storageAccountId={STORAGE_RESOURCE_ID}",
    "--parameters", f"aiServicesId={AI_SERVICES_RESOURCE_ID}",
    "--parameters", f"docIntelligenceId={DI_RESOURCE_ID}",
    "--output", "table",
], capture_output=True, text=True)
print(spl_result.stdout)
if spl_result.returncode != 0:
    print("ERROR (이미 배포됨이면 정상):", spl_result.stderr[:500])

### 3b. SPL 승인

SPL 배포 후 약 5~10분 대기 후, 각 리소스에서 Pending 연결 요청을 자동 승인합니다.

In [ ]:
# SPL 상태 확인
print(f"AI Search: {SEARCH_NAME}")
!az search shared-private-link-resource list \
    --service-name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --query "[].{{name:name, status:properties.status}}" \
    --output table

In [ ]:
# Storage Account SPL 승인
result = subprocess.run(
    ["az", "network", "private-endpoint-connection", "list",
     "--name", STORAGE_NAME,
     "--resource-group", RG_NAME,
     "--type", "Microsoft.Storage/storageAccounts",
     "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
     "--output", "tsv"],
    capture_output=True, text=True
)
pending = [c for c in result.stdout.strip().split('\n') if c]

for conn in pending:
    print(f"승인: {conn}")
    subprocess.run(
        ["az", "network", "private-endpoint-connection", "approve",
         "--name", conn,
         "--resource-name", STORAGE_NAME,
         "--resource-group", RG_NAME,
         "--type", "Microsoft.Storage/storageAccounts",
         "--description", "Approved for AI Search indexer",
         "--output", "none"],
    )

if not pending:
    print("승인 대기 중인 Storage 연결 없음 (이미 승인됨 또는 아직 생성 중)")

In [ ]:
# AI Services + Document Intelligence SPL 승인
# Korea: DI는 별도 RG에 있으므로 리소스별 RG 매핑
spl_targets = [
    (AI_SERVICES_NAME, RG_NAME),
    (DI_NAME, DI_RG),
]

for resource_name, rg in spl_targets:
    result = subprocess.run(
        ["az", "network", "private-endpoint-connection", "list",
         "--name", resource_name,
         "--resource-group", rg,
         "--type", "Microsoft.CognitiveServices/accounts",
         "--query", "[?properties.privateLinkServiceConnectionState.status=='Pending'].name",
         "--output", "tsv"],
        capture_output=True, text=True
    )
    for conn in [c for c in result.stdout.strip().split('\n') if c]:
        print(f"승인: {resource_name} / {conn} (RG: {rg})")
        subprocess.run(
            ["az", "network", "private-endpoint-connection", "approve",
             "--name", conn,
             "--resource-name", resource_name,
             "--resource-group", rg,
             "--type", "Microsoft.CognitiveServices/accounts",
             "--description", "Approved for AI Search skillset",
             "--output", "none"],
        )

In [ ]:
# 최종 승인 상태 확인 — 모두 Approved 여야 AI Search 파이프라인 작동
!az search shared-private-link-resource list \
    --service-name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --query "[].{{name:name, status:properties.status, groupId:properties.groupId}}" \
    --output table

## 4. Function App 코드 배포

Bicep으로 인프라만 생성했으므로, 크롤러 + 스킬 코드(`crawl-function/`)를 Function App에 배포합니다.

In [ ]:
FUNC_APP_NAME = outputs['crawlFunctionName']['value']
print(f"Function App: {FUNC_APP_NAME}")

# Azure Functions Core Tools로 배포
result = subprocess.run(
    ["func", "azure", "functionapp", "publish", FUNC_APP_NAME, "--python"],
    capture_output=True, text=True,
    cwd="../crawl-function"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
import requests as req

# Function App 엔드포인트 확인 (Python v2는 cold start 전 CLI 빈 결과 가능)
result = subprocess.run(
    ["az", "functionapp", "function", "list",
     "--name", FUNC_APP_NAME,
     "--resource-group", RG_NAME,
     "--output", "json"],
    capture_output=True, text=True
)
funcs = json.loads(result.stdout) if result.stdout.strip() else []

if funcs:
    for f in funcs:
        short_name = f["name"].split("/")[-1]
        print(f"  ✅ {short_name}  →  {f.get('invokeUrlTemplate', '')}")
else:
    print("CLI 조회 결과 없음 (Python v2 cold start 전 정상). HTTP 엔드포인트 확인 중...")
    crawl_url = f"https://{FUNC_APP_NAME}.azurewebsites.net/api/crawl"
    try:
        r = req.get(crawl_url, timeout=30)
        print(f"  HTTP {r.status_code} — Function 응답 확인 ({'정상' if r.status_code < 500 else '오류'})")
    except Exception as e:
        print(f"  연결 실패: {e}")

## 5. AI Search 파이프라인 설정

AI Search 네이티브 Indexer + Skillset을 생성합니다:
- **Data Source**: `raw-documents` Blob 컨테이너 감시
- **Skillset**: SplitSkill → AzureOpenAIEmbeddingSkill
- **Index**: `law-documents-index` (벡터 3072D + 시맨틱)
- **Indexer**: 매일 06:00 UTC 자동 실행

> **Note**: AI Search가 Private Network 내에 있으므로, 임시로 Public Access를 허용합니다.

In [ ]:
# AI Search 임시 Public Access 허용
!az search service update \
    --name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --public-network-access enabled \
    --output none

print("AI Search public access 임시 허용")

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

SEARCH_ENDPOINT = outputs['aiSearchEndpoint']['value']
OPENAI_ENDPOINT = outputs['openaiEndpoint']['value']
OPENAI_ACCOUNT  = outputs['openaiAccountName']['value']
STORAGE_ACCOUNT = outputs['storageAccountName']['value']

# 환경변수 세팅 후 파이프라인 설정 스크립트 실행
env = {
    **os.environ,
    "AZURE_SEARCH_SERVICE_ENDPOINT": SEARCH_ENDPOINT,
    "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "AZURE_OPENAI_ACCOUNT_NAME": OPENAI_ACCOUNT,
    "AZURE_STORAGE_ACCOUNT_NAME": STORAGE_ACCOUNT,
    "AZURE_STORAGE_RESOURCE_ID": STORAGE_RESOURCE_ID,
    "AZURE_RESOURCE_GROUP": RG_NAME,
}

result = subprocess.run(
    [sys.executable, "../scripts/setup_ai_search_pipeline.py"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
# 인덱서 즉시 실행 (초기 인덱싱)
result = subprocess.run(
    [sys.executable, "../scripts/setup_ai_search_pipeline.py", "--run"],
    capture_output=True, text=True, env=env
)
print(result.stdout)

### 5b. 멀티모달 파이프라인 등록

PDF/이미지 문서를 위한 두 가지 멀티모달 인덱싱 파이프라인을 등록합니다:
- **[Basic]**: DI Layout → Native SplitSkill (markdown) → Embedding  
- **[Verbalized]**: DI Layout → GPT-5.4 이미지 설명 → Markdown 헤더 분할 → Embedding

> `skills-function` (func-preprocess) 배포가 선행되어야 합니다.

In [ ]:
# skills-function 배포 (preprocess Function App에 skills-function 코드 배포)
import subprocess, os

_suffix = STORAGE_NAME.replace('stragi', '')
SKILLS_FUNC_APP = f"func-preprocess-ragi-{_suffix}"

skills_func_dir = os.path.abspath('../skills-function')
print(f"Deploying skills-function to {SKILLS_FUNC_APP}...")

deploy_result = subprocess.run(
    ["func", "azure", "functionapp", "publish", SKILLS_FUNC_APP, "--python"],
    capture_output=True, text=True, cwd=skills_func_dir
)
print(deploy_result.stdout[-500:] if len(deploy_result.stdout) > 500 else deploy_result.stdout)
if deploy_result.returncode != 0:
    print("ERROR:", deploy_result.stderr[-500:])

In [ ]:
# Skills Function Key 조회
import json, subprocess

key_result = subprocess.run(
    ["az", "functionapp", "keys", "list",
     "--name", SKILLS_FUNC_APP,
     "--resource-group", RG_NAME,
     "--query", "functionKeys.default",
     "--output", "tsv"],
    capture_output=True, text=True
)
SKILLS_FUNCTION_KEY = key_result.stdout.strip()
SKILLS_FUNCTION_URL = f"https://{SKILLS_FUNC_APP}.azurewebsites.net"

print(f"Skills Function URL: {SKILLS_FUNCTION_URL}")
print(f"Skills Function Key: {SKILLS_FUNCTION_KEY[:12]}..." if SKILLS_FUNCTION_KEY else "ERROR: Key not found")

In [ ]:
# 멀티모달 파이프라인 등록 (Basic + Verbalized)
import subprocess, sys, os

mm_env = {
    **os.environ,
    "AZURE_SEARCH_SERVICE_ENDPOINT": SEARCH_ENDPOINT,
    "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "AZURE_STORAGE_RESOURCE_ID": STORAGE_RESOURCE_ID,
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT": "text-embedding-3-large",
    "AZURE_STORAGE_CONTAINER_NAME": "raw-documents",
    "SKILLS_FUNCTION_URL": SKILLS_FUNCTION_URL,
    "SKILLS_FUNCTION_KEY": SKILLS_FUNCTION_KEY,
}

result = subprocess.run(
    [sys.executable, "../scripts/setup_ai_search_multimodal_pipeline.py",
     "--source", "st", "--pipeline", "both"],
    capture_output=True, text=True, env=mm_env
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr[-1000:])

In [ ]:
# 멀티모달 파이프라인 등록 확인
from azure.identity import DefaultAzureCredential
import requests

credential = DefaultAzureCredential()
token = credential.get_token("https://search.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

for resource, names in [
    ("indexes", ["st-multimodal-basic-index", "st-multimodal-verbalized-index"]),
    ("indexers", ["st-multimodal-basic-indexer", "st-multimodal-verbalized-indexer"]),
    ("skillsets", ["st-multimodal-basic-skillset", "st-multimodal-verbalized-skillset"]),
]:
    for name in names:
        r = requests.get(f"{SEARCH_ENDPOINT}/{resource}/{name}?api-version=2024-07-01", headers=headers)
        status = "✓" if r.status_code == 200 else f"✗ ({r.status_code})"
        print(f"  {status} {resource}/{name}")

### 5c. Knowledge Agent + 4개 Knowledge Source 생성

`04-search-and-query.ipynb` 의 시나리오 D/E/F (agentic retrieval) 가 사용하는 다음을 사전 등록합니다:

- **4개 Knowledge Source** (`ks-prec`, `ks-const`, `ks-interp`, `ks-admin`) ← 4개 법령 인덱스 매핑
- **1개 Knowledge Agent** (`legal-knowledge-agent`) ← `gpt-4o` planner + 4개 source 통합

> 선행 조건: 5번 셀에서 `setup_ai_search_pipeline.py` 가 4개 인덱스를 생성했어야 함.
> AOAI 호출을 위해 Search MI 에 `Cognitive Services OpenAI User` 권한이 부여되어 있어야 함 (Bicep 에 포함됨).


In [ ]:
# Knowledge Sources + Knowledge Agent 생성 (idempotent)
import sys, requests as _req
sys.path.insert(0, os.path.abspath('..'))
from src.search.legal_indexes import PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

KS_API_VERSION = "2025-08-01-preview"
PLANNER_DEPLOY = "gpt-4o"   # AOAI deployment for Knowledge Agent planner
AGENT_NAME     = "legal-knowledge-agent"

cred_ks = DefaultAzureCredential()
def _ks_hdr():
    tok = cred_ks.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

INDEX_TO_SRC = {
    PREC_INDEX:   ("ks-prec",   "한국 법원 판례",
        "caseName,caseNumber,courtName,courtLevel,judgmentDate,relatedLaws,keywords,holdings,summary"),
    CONST_INDEX:  ("ks-const",  "헌법재판소 결정례",
        "caseName,caseNumber,decisionDate,decisionType,relatedLaws,keywords,holdings,summary"),
    INTERP_INDEX: ("ks-interp", "법제처 법령해석례",
        "title,docNumber,replyDate,interpType,relatedLaws,keywords,querySummary,reply"),
    ADMIN_INDEX:  ("ks-admin",  "행정심판 재결례",
        "caseName,caseNumber,decisionDate,decisionType,committee,relatedLaws,keywords,order,reasonSummary"),
}

# 1) Knowledge Source 4개 (PUT = upsert)
for idx_name, (src_name, desc, fields) in INDEX_TO_SRC.items():
    body = {
        "name": src_name, "kind": "searchIndex", "description": desc,
        "searchIndexParameters": {"searchIndexName": idx_name, "sourceDataSelect": fields},
    }
    url = f"{SEARCH_ENDPOINT}/knowledgesources('{src_name}')?api-version={KS_API_VERSION}"
    r = _req.put(url, headers=_ks_hdr(), json=body)
    print(f"  KS '{src_name}' <- {idx_name}: {r.status_code}")
    if r.status_code >= 400: print("    ", r.text[:300])

# 2) Knowledge Agent (PUT = upsert)
agent_body = {
    "name": AGENT_NAME,
    "description": "Korean legal corpus: 판례 / 헌재 / 법제처 / 행심",
    "models": [{"kind": "azureOpenAI", "azureOpenAIParameters": {
        "resourceUri": OPENAI_ENDPOINT.rstrip("/"),
        "deploymentId": PLANNER_DEPLOY, "modelName": PLANNER_DEPLOY,
    }}],
    "knowledgeSources": [
        {"name": s, "alwaysQuerySource": True, "includeReferences": True,
         "includeReferenceSourceData": True, "maxSubQueries": 4, "rerankerThreshold": 1.5}
        for _, (s, _, _) in INDEX_TO_SRC.items()
    ],
    "outputConfiguration": {"modality": "answerSynthesis", "includeActivity": True, "attemptFastPath": False},
    "requestLimits": {"maxRuntimeInSeconds": 60, "maxOutputSize": 16000},
    "retrievalInstructions": (
        "사용자 질문에 한국 법령·판례 자료가 필요하면 4개 소스에서 hybrid+semantic 검색을 수행한다. "
        "이전 대화 메시지에 사건번호(예: 2019도1234)나 법령명이 언급되어 있으면 그 키워드로 sub-query 를 추가 계획하라."
    ),
}
url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')?api-version={KS_API_VERSION}"
r = _req.put(url, headers=_ks_hdr(), json=agent_body)
print(f"\nAgent '{AGENT_NAME}': {r.status_code}")
if r.status_code >= 400:
    print(r.text[:800])
else:
    print(f"  planner = {PLANNER_DEPLOY}")
    print(f"  sources = {[s for _, (s, _, _) in INDEX_TO_SRC.items()]}")


### 5d. Foundry Agent Service RBAC (Sweden 전용)

`04-search-and-query.ipynb` 의 시나리오 E/F (Foundry Agent Service) 가 사용하는 권한을 사전 부여합니다.

| 권한 | Principal | Scope | 용도 |
|---|---|---|---|
| `Azure AI User` | 현재 로그인 사용자 | AIServices 계정 | Foundry project + agent 생성/조회/실행 |

> Korea 리전은 Foundry 미지원 — 자동 skip


In [ ]:
# Foundry RBAC + project endpoint 추출 (Sweden 전용)
FOUNDRY_PROJECT_ENDPOINT = ""
if DEPLOY_REGION == "sweden":
    AIS_RID = subprocess.run(
        ["az", "cognitiveservices", "account", "show",
         "--name", AI_SERVICES_NAME, "--resource-group", RG_NAME,
         "--query", "id", "-o", "tsv"],
        capture_output=True, text=True
    ).stdout.strip()

    # 현재 로그인 사용자 object id
    USER_OID = subprocess.run(
        ["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"current user OID: {USER_OID}")

    # Azure AI User 역할 부여 (idempotent — 이미 있으면 무시)
    azure_ai_user_role = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
    rbac = subprocess.run(
        ["az", "role", "assignment", "create",
         "--assignee", USER_OID,
         "--role", azure_ai_user_role,
         "--scope", AIS_RID,
         "-o", "none"],
        capture_output=True, text=True
    )
    if rbac.returncode == 0:
        print(f"  ✓ Azure AI User granted on {AI_SERVICES_NAME}")
    elif "already exists" in (rbac.stderr or ""):
        print(f"  ♻️  Azure AI User already assigned")
    else:
        print(f"  ⚠️  RBAC stderr: {rbac.stderr[:300]}")

    # Foundry default project endpoint 조회 (Bicep 의 allowProjectManagement=true 로 자동 생성됨)
    proj = subprocess.run(
        ["az", "cognitiveservices", "account", "project", "list",
         "--account-name", AI_SERVICES_NAME, "--resource-group", RG_NAME,
         "--query", "[0].properties.endpoints", "-o", "json"],
        capture_output=True, text=True
    )
    try:
        endpoints = json.loads(proj.stdout) if proj.stdout.strip() else {}
        FOUNDRY_PROJECT_ENDPOINT = endpoints.get("AI Foundry API", "")
        print(f"  Foundry project endpoint: {FOUNDRY_PROJECT_ENDPOINT}")
    except Exception as e:
        print(f"  ⚠️  project endpoint 조회 실패 (수동 확인 필요): {e}")
else:
    print(f"  → {DEPLOY_REGION} 리전: Foundry 미지원, skip")


In [ ]:
# AI Search Public Access 다시 비활성화
!az search service update \
    --name {SEARCH_NAME} \
    --resource-group {RG_NAME} \
    --public-network-access disabled \
    --output none

print("AI Search public access 비활성화 완료")

## 5.5 Skills Function 배포 (Custom Web API Skills)

AI Search 멀티모달 파이프라인이 사용하는 Custom Skills를 배포합니다.

| Skill | Endpoint | 용도 |
|-------|----------|------|
| `markdown_split` | `/api/markdown_split` | PDF Markdown 헤더(`#`,`##`,`###`) + 길이 기반 분할 |
| `pptx_page_split` | `/api/pptx_page_split` | PPTX `<!-- PageBreak -->` 기반 슬라이드 단위 분할 |
| `verbalize` | `/api/verbalize` | GPT-5.4 Vision으로 이미지/도표 설명 생성 |

배포 대상 Function App: `func-preprocess-ragi-{suffix}` (Bicep으로 사전 생성됨, FlexConsumption + VNet)


In [ ]:
# ── Skills Function App 이름 결정 (Bicep output 사용) ──
# preprocessFunctionName output이 있으면 사용, 없으면 stable suffix 기반 fallback
SKILLS_FUNC_APP = (
    outputs.get('preprocessFunctionName', {}).get('value')
    or f"func-preprocess-ragi-{outputs.get('storageAccountName', {}).get('value', '')[-8:]}"
)
print(f"Target Function App: {SKILLS_FUNC_APP}")

# Function App 존재 확인
check = subprocess.run(
    ["az", "functionapp", "show", "--name", SKILLS_FUNC_APP, "--resource-group", RG_NAME, "--query", "state", "-o", "tsv"],
    capture_output=True, text=True
)
assert check.returncode == 0, f"Function App not found: {SKILLS_FUNC_APP}\n{check.stderr}"
print(f"  state: {check.stdout.strip()}")


In [ ]:
# ── App Settings 주입 (OpenAI/DI 엔드포인트 + Managed Identity 인증용) ──
APP_SETTINGS = [
    f"AZURE_OPENAI_ENDPOINT={OPENAI_ENDPOINT}",
    f"AZURE_OPENAI_GPT54_DEPLOYMENT={GPT_DEPLOY_NAME}",
    f"AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={DI_ENDPOINT}",
]
print("Setting app settings...")
result = subprocess.run(
    ["az", "functionapp", "config", "appsettings", "set",
     "--name", SKILLS_FUNC_APP, "--resource-group", RG_NAME,
     "--settings", *APP_SETTINGS,
     "-o", "none"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"WARN: appsettings set failed: {result.stderr}")
else:
    print("  ✓ app settings configured")


In [ ]:
# ── skills-function 코드 배포 (func azure functionapp publish) ──
skills_dir = os.path.abspath('../skills-function')
print(f"Deploying skills-function from {skills_dir} → {SKILLS_FUNC_APP}")

result = subprocess.run(
    ["func", "azure", "functionapp", "publish", SKILLS_FUNC_APP, "--python", "--build", "remote"],
    cwd=skills_dir, capture_output=True, text=True, timeout=900
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"func publish failed (exit={result.returncode})")
print("✓ skills-function 배포 완료")


In [ ]:
# ── Function key + URL 추출 (.env 셀에서 사용) ──
key_result = subprocess.run(
    ["az", "functionapp", "keys", "list",
     "--name", SKILLS_FUNC_APP, "--resource-group", RG_NAME,
     "--query", "functionKeys.default", "-o", "tsv"],
    capture_output=True, text=True
)
SKILLS_FUNCTION_KEY = key_result.stdout.strip()
SKILLS_FUNCTION_URL = f"https://{SKILLS_FUNC_APP}.azurewebsites.net"

print(f"Skills Function URL: {SKILLS_FUNCTION_URL}")
print(f"Skills Function Key: {SKILLS_FUNCTION_KEY[:12]}..." if SKILLS_FUNCTION_KEY else "ERROR: Key not found")


## 6. .env 파일 생성

배포 출력값으로 `.env` 파일을 자동 생성합니다.  
모든 인증은 **Managed Identity (DefaultAzureCredential)** 기반으로, API 키가 필요 없습니다.

In [ ]:
CRAWL_FUNC_URL  = outputs.get('crawlFunctionUrl', {}).get('value', '')
LOGIC_APP_NAME  = outputs.get('crawlLogicAppName', {}).get('value', '')
DI_ENDPOINT     = outputs['docIntelligenceEndpoint']['value']

env_content = f"""# Auto-generated from Bicep deployment — DO NOT commit to git
AZURE_SUBSCRIPTION_ID={SUBSCRIPTION_ID}
AZURE_RESOURCE_GROUP={RG_NAME}
AZURE_LOCATION={LOCATION}
DEPLOY_REGION={DEPLOY_REGION}

# Storage (Private — Managed Identity 인증)
AZURE_STORAGE_ACCOUNT_NAME={STORAGE_ACCOUNT}
AZURE_STORAGE_CONTAINER_NAME=raw-documents
AZURE_STORAGE_RESOURCE_ID={STORAGE_RESOURCE_ID}
BLOB_CONTAINER_NAME=raw-documents

# Azure AI Services (OpenAI)
AZURE_OPENAI_ENDPOINT={OPENAI_ENDPOINT}
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AZURE_OPENAI_GPT54_DEPLOYMENT={GPT_DEPLOY_NAME}

# Azure AI Search
AZURE_SEARCH_SERVICE_ENDPOINT={SEARCH_ENDPOINT}
AZURE_SEARCH_INDEX_NAME=law-documents-index

# Document Intelligence
AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT={DI_ENDPOINT}

# Crawl Pipeline
AZURE_FUNCTION_CRAWL_URL={CRAWL_FUNC_URL}
AZURE_LOGIC_APP_NAME={LOGIC_APP_NAME}

# Skills Function (Multimodal Pipeline)
SKILLS_FUNCTION_URL={SKILLS_FUNCTION_URL}
SKILLS_FUNCTION_KEY={SKILLS_FUNCTION_KEY}

# Foundry Agent Service (Sweden 전용 — Knowledge Agent + agent demo)
FOUNDRY_PROJECT_ENDPOINT={FOUNDRY_PROJECT_ENDPOINT}
KNOWLEDGE_AGENT_NAME={AGENT_NAME}
"""

with open('../.env', 'w') as f:
    f.write(env_content)

print(".env 파일 생성 완료 (Managed Identity 기준 변수 포함)")
print(env_content)


## 7. 배포 검증

In [ ]:
# 리소스 그룹 내 전체 리소스 확인
!az resource list --resource-group {RG_NAME} --output table

# Korea: DI 별도 RG 리소스도 확인
if cfg["has_di_rg"]:
    print(f"\n=== DI RG ({DI_RG}) ===")
    !az resource list --resource-group {DI_RG} --output table

In [ ]:
import requests

# Function App 수동 크롤 테스트 (limit=2로 빠르게)
CRAWL_FUNC_URL = outputs.get('crawlFunctionUrl', {}).get('value', f"https://{FUNC_APP_NAME}.azurewebsites.net/api/crawl")
resp = requests.post(
    CRAWL_FUNC_URL,
    json={"limit": 2, "triggered_by": "notebook-test"},
    timeout=120
)
print(f"Status: {resp.status_code}")
print(resp.json())

In [ ]:
# AI Search 인덱서 상태 확인
from azure.identity import DefaultAzureCredential
import requests as req

credential = DefaultAzureCredential()
token = credential.get_token("https://search.azure.com/.default").token

r = req.get(
    f"{SEARCH_ENDPOINT}/indexers/law-blob-indexer/status?api-version=2024-07-01",
    headers={"Authorization": f"Bearer {token}"}
)
status = r.json()
last_run = status.get("lastResult", {})
print(f"인덱서 상태: {last_run.get('status', 'N/A')}")
print(f"처리된 문서: {last_run.get('itemsProcessed', 0)}")
print(f"실패한 문서: {last_run.get('itemsFailed', 0)}")
print(f"완료 시각: {last_run.get('endTime', 'N/A')}")

---
다음 단계: [02-data-crawling.ipynb](02-data-crawling.ipynb) — Azure Function App으로 크롤링 실행 및 모니터링